# 05 — Generator Alignment (DPO)

This notebook runs **DPO alignment** on the generator using preference pairs from `data/processed/dpo_pairs.jsonl`.

- Starts from the **DoRA adapter checkpoint** in `models/generator_dora/`.
- Saves aligned adapters to `models/generator_dpo/`.

**Input schema (JSONL)**: each row must contain `prompt`, `chosen`, `rejected`.


In [ ]:
import os
import sys
import json
from pathlib import Path

print('Python:', sys.version)
print('CWD:', os.getcwd())

ROOT = Path('.').resolve()
print('ROOT:', ROOT)

pairs_path = ROOT / 'data' / 'processed' / 'dpo_pairs.jsonl'
print('pairs_path exists:', pairs_path.exists(), str(pairs_path))
assert pairs_path.exists(), 'Missing data/processed/dpo_pairs.jsonl'

# Preview a couple rows
with pairs_path.open('r', encoding='utf-8') as f:
    for i in range(2):
        obj = json.loads(f.readline())
        print('--- row', i)
        print('keys:', list(obj.keys()))
        print('prompt chars:', len(obj.get('prompt','')))
        print('chosen chars:', len(obj.get('chosen','')))
        print('rejected chars:', len(obj.get('rejected','')))

adapter_dir = ROOT / 'models' / 'generator_dora'
print('adapter_dir exists:', adapter_dir.exists(), str(adapter_dir))
assert adapter_dir.exists(), 'Missing models/generator_dora (run Notebook 04 first)'


## Run DPO training

Defaults are conservative for an L4. If you hit OOM, add `--load_in_4bit` and/or reduce `--max_length`.


In [ ]:
!python -u src/generator/train_dpo.py \
  --pairs data/processed/dpo_pairs.jsonl \
  --sft_adapter models/generator_dora \
  --out models/generator_dpo \
  --epochs 1 \
  --batch_size 1 \
  --grad_accum 16 \
  --lr 5e-6 \
  --max_length 1024 \
  --beta 0.1 \
  --bf16

In [ ]:
from pathlib import Path

out_dir = Path('models/generator_dpo')
print('out_dir exists:', out_dir.exists())
if out_dir.exists():
    for p in sorted(out_dir.glob('*'))[:80]:
        print(p.name)

expected_any = [
    'adapter_config.json',
    'adapter_model.safetensors',
    'adapter_model.bin',
    'tokenizer.json',
    'tokenizer_config.json',
]
found = {p.name for p in out_dir.glob('*')} if out_dir.exists() else set()
print('found expected:', {x: (x in found) for x in expected_any})


⏸️ TRAINING CHECKPOINT

Before proceeding to the tool-policy model and ReAct loop, confirm that `models/generator_dpo/` contains:
- `adapter_config.json`
- adapter weights (`adapter_model.safetensors` or `.bin`)
- tokenizer files

Reply with the output of `ls -la models/generator_dpo` from Lightning AI.